# CLIP: Text-to-Image Shoe Search

This notebook uses OpenAI's **CLIP** (Contrastive Language-Image Pretraining) model to search for shoes by text description.

**What makes CLIP special:**
- Understands both images and text in the same embedding space
- No training needed — works out of the box (zero-shot)
- You can search using natural language: "red sneakers", "black boots for winter", etc.
- Runs 100% locally, no API keys or internet required after setup

**What this notebook does:**
1. Loads CLIP model (ViT-B/32)
2. Extracts image embeddings for all shoes in the dataset
3. Saves embeddings for fast reuse
4. Provides a text search interface to find shoes by description

**Prerequisites:**
- `raw_data/images_filtered/` folder with product images (created by `kyrylo_test.ipynb`)
- `raw_data/articles_filtered.csv` with product metadata
- Required packages: `pip install openai-clip torch Pillow matplotlib tqdm`

## Step 1: Install CLIP (run once)

CLIP is an open-source model by OpenAI. It does NOT require an API key or ChatGPT — everything runs locally on your machine.

In [ ]:
# Uncomment and run this cell once to install CLIP
# !pip install openai-clip torch torchvision tqdm

## Step 2: Import Libraries

We use:
- `clip` — OpenAI's CLIP model for image and text embeddings
- `torch` — PyTorch backend for CLIP
- `PIL` — image loading
- `numpy` — similarity calculations
- `pandas` — product metadata
- `matplotlib` — displaying results

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# CLIP and PyTorch
import torch
import clip

# Check device (MPS for Apple Silicon, CUDA for NVIDIA, else CPU)
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print(f"PyTorch version: {torch.__version__}")
print(f"CLIP version: {clip.__version__ if hasattr(clip, '__version__') else 'installed'}")
print(f"Device: {device}")

## Step 3: Configuration

Define all paths and parameters in one place.

In [ ]:
# Paths
IMAGES_DIR = '../raw_data/images_filtered'
ARTICLES_CSV = '../raw_data/articles_filtered.csv'
EMBEDDINGS_DIR = '../embeddings/clip'
EMBEDDINGS_FILE = os.path.join(EMBEDDINGS_DIR, 'clip_image_embeddings.pkl')

# CLIP model variant
CLIP_MODEL = 'ViT-B/32'  # Good balance of speed and quality

# Search parameters
TOP_K = 5  # Number of results to return

# Create embeddings directory
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
print(f"Images: {IMAGES_DIR}")
print(f"Embeddings: {EMBEDDINGS_FILE}")
print(f"CLIP model: {CLIP_MODEL}")

## Step 4: Load CLIP Model

Load the pretrained CLIP model. On first run, it downloads the weights (~350MB).

**Model variants:**
- `ViT-B/32` — fast, good quality (we use this)
- `ViT-B/16` — slower, better quality
- `ViT-L/14` — slowest, best quality

In [ ]:
# Load CLIP model and image preprocessing function
model, preprocess = clip.load(CLIP_MODEL, device=device)
model.eval()  # Set to evaluation mode

print(f"Model loaded: {CLIP_MODEL}")
print(f"Image input size: {model.visual.input_resolution}")
print(f"Embedding dimension: {model.visual.output_dim}")

## Step 5: Load Product Metadata

Load article information for displaying search results.

In [ ]:
# Load articles data
df_articles = pd.read_csv(ARTICLES_CSV)
print(f"Total articles: {len(df_articles)}")

# Select relevant columns
info_columns = ['article_id', 'prod_name', 'product_type_name', 'colour_group_name',
                'department_name', 'index_group_name', 'detail_desc']
df_articles_info = df_articles[info_columns].copy()

# Load image file list
image_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')])
print(f"Total images: {len(image_files)}")

## Step 6: Extract Image Embeddings with CLIP

CLIP converts each image into a 512-dimensional vector that captures its visual meaning.
These vectors live in the same space as text vectors, which enables text-to-image search.

**Batch processing** is used for faster extraction.

In [ ]:
def extract_clip_embeddings(image_files, images_dir, model, preprocess, device, batch_size=32, save_path=None):
    """
    Extract CLIP embeddings for all images using batch processing.
    
    Args:
        image_files: List of image filenames
        images_dir: Directory containing images
        model: CLIP model
        preprocess: CLIP image preprocessing function
        device: torch device (cpu/mps/cuda)
        batch_size: Number of images to process at once
        save_path: Optional path to save embeddings
    
    Returns:
        Dictionary with 'embeddings' (numpy array) and 'filenames' (list)
    """
    all_embeddings = []
    valid_files = []
    
    # Process in batches
    for i in tqdm(range(0, len(image_files), batch_size), desc="Extracting CLIP embeddings"):
        batch_files = image_files[i:i + batch_size]
        batch_images = []
        batch_valid = []
        
        for filename in batch_files:
            try:
                img_path = os.path.join(images_dir, filename)
                img = preprocess(Image.open(img_path).convert('RGB'))
                batch_images.append(img)
                batch_valid.append(filename)
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        
        if batch_images:
            # Stack into tensor and move to device
            image_tensor = torch.stack(batch_images).to(device)
            
            # Extract features (no gradient needed)
            with torch.no_grad():
                features = model.encode_image(image_tensor)
                # Normalize embeddings
                features = features / features.norm(dim=-1, keepdim=True)
            
            all_embeddings.append(features.cpu().numpy())
            valid_files.extend(batch_valid)
    
    result = {
        'embeddings': np.vstack(all_embeddings),
        'filenames': valid_files
    }
    
    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump(result, f)
        print(f"Embeddings saved to: {save_path}")
    
    return result

In [ ]:
# Load existing embeddings or extract new ones
if os.path.exists(EMBEDDINGS_FILE):
    print(f"Loading existing CLIP embeddings from: {EMBEDDINGS_FILE}")
    with open(EMBEDDINGS_FILE, 'rb') as f:
        embeddings_data = pickle.load(f)
    print(f"Loaded {len(embeddings_data['filenames'])} embeddings")
else:
    print("Extracting CLIP embeddings for all images...")
    print("This is faster than ResNet50 thanks to batch processing.\n")
    embeddings_data = extract_clip_embeddings(
        image_files,
        IMAGES_DIR,
        model,
        preprocess,
        device,
        batch_size=64,
        save_path=EMBEDDINGS_FILE
    )

# Unpack
image_embeddings = embeddings_data['embeddings']
filenames = embeddings_data['filenames']
print(f"\nEmbeddings shape: {image_embeddings.shape}")
print(f"Number of images: {len(filenames)}")

## Step 7: Define Text-to-Image Search Function

CLIP's key feature: it encodes text and images into the **same vector space**.
So we can compute similarity between a text query and all image embeddings directly.

In [ ]:
def filename_to_article_id(filename):
    """Convert filename like '0181160009.jpg' to article_id 181160009"""
    try:
        return int(filename.replace('.jpg', ''))
    except ValueError:
        return None


def get_article_info(filename, df_articles):
    """
    Get product info for a given image filename.
    Returns dictionary with product details or None.
    """
    article_id = filename_to_article_id(filename)
    if article_id is None:
        return None
    
    article_row = df_articles[df_articles['article_id'] == article_id]
    if len(article_row) == 0:
        return None
    
    row = article_row.iloc[0]
    return {
        'article_id': article_id,
        'name': row['prod_name'],
        'type': row['product_type_name'],
        'color': row['colour_group_name'],
        'department': row['department_name'],
        'category': row['index_group_name'],
        'description': row.get('detail_desc', '')
    }


def text_search(query, model, image_embeddings, filenames, device, top_k=TOP_K):
    """
    Search for images using a text query.
    
    Args:
        query: Text description (e.g., "red sneakers")
        model: CLIP model
        image_embeddings: Precomputed image embeddings (normalized)
        filenames: List of image filenames
        device: torch device
        top_k: Number of results
    
    Returns:
        List of (filename, similarity_score) tuples
    """
    # Encode text query
    text_tokens = clip.tokenize([query]).to(device)
    with torch.no_grad():
        text_embedding = model.encode_text(text_tokens)
        text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
    
    text_embedding = text_embedding.cpu().numpy().flatten()
    
    # Calculate similarities (embeddings are already normalized)
    similarities = np.dot(image_embeddings, text_embedding)
    
    # Get top-k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = [(filenames[idx], similarities[idx]) for idx in top_indices]
    return results

## Step 8: Define Display Function

Show search results as a visual grid with product details.

In [ ]:
def display_text_search_results(query, results, images_dir, df_articles):
    """
    Display text search results with images and product info.
    
    Args:
        query: The text query
        results: List of (filename, similarity) tuples
        images_dir: Directory with images
        df_articles: Articles DataFrame
    """
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
    
    if n == 1:
        axes = [axes]
    
    fig.suptitle(f'Search: \"{query}\"', fontsize=14, fontweight='bold', color='blue', y=1.02)
    
    for i, (filename, similarity) in enumerate(results):
        img_path = os.path.join(images_dir, filename)
        img = Image.open(img_path)
        
        axes[i].imshow(img)
        axes[i].axis('off')
        
        info = get_article_info(filename, df_articles)
        if info:
            title = f"#{i+1} | Score: {similarity:.3f}"
            label = f"{info['name']}\n{info['type']} | {info['color']}\n{info['category']}"
        else:
            title = f"#{i+1} | Score: {similarity:.3f}"
            label = filename
        
        axes[i].set_title(title, fontsize=10)
        axes[i].set_xlabel(label, fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed info
    print("\n" + "="*80)
    print(f"SEARCH RESULTS for: \"{query}\"")
    print("="*80)
    for i, (filename, similarity) in enumerate(results):
        info = get_article_info(filename, df_articles)
        print(f"\n#{i+1} | Score: {similarity:.4f}")
        if info:
            print(f"   Article ID: {info['article_id']}")
            print(f"   Name: {info['name']}")
            print(f"   Type: {info['type']}")
            print(f"   Color: {info['color']}")
            print(f"   Department: {info['department']}")
            print(f"   Category: {info['category']}")
            if info['description']:
                print(f"   Description: {info['description'][:100]}...")

## Step 9: Main Search Function

Combines text encoding, similarity search, and result display.

In [ ]:
def search_shoes(query, top_k=TOP_K):
    """
    Search for shoes by text description.
    
    Args:
        query: Natural language description (e.g., "red sneakers", "elegant black heels")
        top_k: Number of results to show
    """
    print(f"Searching for: \"{query}\"")
    print("-" * 50)
    
    results = text_search(query, model, image_embeddings, filenames, device, top_k=top_k)
    display_text_search_results(query, results, IMAGES_DIR, df_articles_info)
    
    return results

## Step 10: Demo — Try Different Queries

Let's test CLIP with different types of text queries to see how well it understands shoe descriptions.

In [ ]:
# Search by color + type
search_shoes("red sneakers")

In [ ]:
# Search by style
search_shoes("elegant black high heels")

In [ ]:
# Search by occasion
search_shoes("casual summer sandals")

In [ ]:
# Search by material/feature
search_shoes("leather boots")

## Step 11: Interactive Search

Type your own query to search for shoes! Run this cell and enter your text.

In [ ]:
# Type your search query below
query = input("Describe the shoes you are looking for: ")

if query.strip():
    search_shoes(query.strip())
else:
    print("No query entered. Try something like:")
    print('  - "white running shoes"')
    print('  - "pink ballet flats"')
    print('  - "warm winter boots"')

## Step 12: Multi-Query Comparison

Compare results for multiple queries side by side. Useful for understanding how CLIP interprets different descriptions.

In [ ]:
def compare_queries(queries, top_k=3):
    """
    Show top results for multiple queries side by side.
    
    Args:
        queries: List of text queries to compare
        top_k: Number of results per query
    """
    n_queries = len(queries)
    fig, axes = plt.subplots(n_queries, top_k, figsize=(4 * top_k, 4 * n_queries))
    
    if n_queries == 1:
        axes = [axes]
    
    for row, query in enumerate(queries):
        results = text_search(query, model, image_embeddings, filenames, device, top_k=top_k)
        
        for col, (filename, similarity) in enumerate(results):
            img_path = os.path.join(IMAGES_DIR, filename)
            img = Image.open(img_path)
            
            axes[row][col].imshow(img)
            axes[row][col].axis('off')
            
            info = get_article_info(filename, df_articles_info)
            if col == 0:
                axes[row][col].set_ylabel(f'\"{query}\"', fontsize=11, fontweight='bold', rotation=0, labelpad=120)
            
            name = info['name'] if info else filename
            axes[row][col].set_title(f"{similarity:.3f} — {name}", fontsize=9)
    
    plt.suptitle('Multi-Query Comparison', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Compare different shoe types
compare_queries([
    "white sneakers",
    "black boots",
    "pink sandals",
    "brown leather shoes"
])

## Step 13: Image-to-Image Search with CLIP

CLIP can also do image-to-image search (like ResNet50 in the previous notebook), but with CLIP embeddings.

In [ ]:
def image_search(query_image_path, top_k=TOP_K):
    """
    Search for similar shoes using an image query.
    
    Args:
        query_image_path: Path to query image
        top_k: Number of results
    """
    # Load and preprocess query image
    img = preprocess(Image.open(query_image_path).convert('RGB')).unsqueeze(0).to(device)
    
    with torch.no_grad():
        query_embedding = model.encode_image(img)
        query_embedding = query_embedding / query_embedding.norm(dim=-1, keepdim=True)
    
    query_embedding = query_embedding.cpu().numpy().flatten()
    
    # Find similar images
    similarities = np.dot(image_embeddings, query_embedding)
    top_indices = np.argsort(similarities)[::-1]
    
    # Exclude exact match
    results = []
    for idx in top_indices:
        if similarities[idx] > 0.9999:
            continue
        results.append((filenames[idx], similarities[idx]))
        if len(results) >= top_k:
            break
    
    # Display
    n = len(results) + 1
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
    
    # Show query image
    query_img = Image.open(query_image_path)
    axes[0].imshow(query_img)
    axes[0].set_title('QUERY IMAGE', fontsize=12, fontweight='bold', color='blue')
    axes[0].axis('off')
    
    for i, (filename, sim) in enumerate(results):
        img = Image.open(os.path.join(IMAGES_DIR, filename))
        axes[i+1].imshow(img)
        axes[i+1].axis('off')
        info = get_article_info(filename, df_articles_info)
        name = info['name'] if info else filename
        axes[i+1].set_title(f"#{i+1} | {sim:.3f}", fontsize=10)
        axes[i+1].set_xlabel(f"{name}\n{info['type']} | {info['color']}" if info else filename, fontsize=9)
    
    plt.tight_layout()
    plt.show()
    return results

# Test with image from test_images
test_image = '../raw_data/test_images/test_shoe.jpg'
if os.path.exists(test_image):
    print(f"Image search with: {test_image}\n")
    image_search(test_image)
else:
    # Use a random image from dataset
    random_file = filenames[42]
    random_path = os.path.join(IMAGES_DIR, random_file)
    print(f"Image search with dataset image: {random_file}\n")
    image_search(random_path)

## Step 14: Combined Search — Text + Image Filter

Advanced feature: search by text but filter results by product type.

**Example:** Find "red shoes" but only show sneakers.

In [ ]:
def filtered_search(query, product_type=None, color=None, top_k=TOP_K):
    """
    Search with text query and optional filters.
    
    Args:
        query: Text description
        product_type: Filter by type (e.g., 'Sneakers', 'Boots')
        color: Filter by color (e.g., 'Black', 'White')
        top_k: Number of results
    """
    # Encode text
    text_tokens = clip.tokenize([query]).to(device)
    with torch.no_grad():
        text_embedding = model.encode_text(text_tokens)
        text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
    text_embedding = text_embedding.cpu().numpy().flatten()
    
    # Calculate similarities
    similarities = np.dot(image_embeddings, text_embedding)
    
    # Sort by similarity
    sorted_indices = np.argsort(similarities)[::-1]
    
    # Filter results
    results = []
    for idx in sorted_indices:
        filename = filenames[idx]
        info = get_article_info(filename, df_articles_info)
        
        if info is None:
            continue
        
        # Apply filters
        if product_type and info['type'] != product_type:
            continue
        if color and info['color'] != color:
            continue
        
        results.append((filename, similarities[idx]))
        if len(results) >= top_k:
            break
    
    # Display
    filter_text = ""
    if product_type:
        filter_text += f" | Type: {product_type}"
    if color:
        filter_text += f" | Color: {color}"
    
    print(f"Query: \"{query}\"{filter_text}")
    print(f"Found {len(results)} results\n")
    display_text_search_results(query + filter_text, results, IMAGES_DIR, df_articles_info)
    return results

# Example: search for comfortable shoes, only sneakers
filtered_search("comfortable casual shoes", product_type="Sneakers")

In [ ]:
# Example: black boots only
filtered_search("stylish winter footwear", product_type="Boots", color="Black")

## Step 15: Interactive Search with Input Field

Enter your query and optional filters.

In [ ]:
# Available product types and colors for reference
print("Available product types:")
print(df_articles['product_type_name'].value_counts().to_string())
print("\nTop colors:")
print(df_articles['colour_group_name'].value_counts().head(10).to_string())

In [ ]:
# Interactive search with filters
query = input("Describe the shoes: ")
product_type = input("Filter by type (leave empty for all): ").strip() or None
color = input("Filter by color (leave empty for all): ").strip() or None

if query.strip():
    filtered_search(query.strip(), product_type=product_type, color=color)
else:
    print("No query entered.")

## Summary

This notebook implements a **CLIP-based shoe search system** with three search modes:

| Mode | Description | Example |
|------|-------------|---------|
| **Text search** | Find shoes by description | `"red sneakers"` |
| **Image search** | Find visually similar shoes | Upload any shoe image |
| **Filtered search** | Text + category/color filter | `"casual shoes"` + type=Sneakers |

**Key advantages over ResNet50 (kyrylo_test2):**
- Natural language search (ResNet50 can only do image-to-image)
- Zero-shot — no training needed
- Multi-modal — understands both text and images
- Faster batch processing with PyTorch

**Possible improvements:**
- Fine-tune CLIP on shoe-specific data for better accuracy
- Add multilingual support (CLIP supports multiple languages)
- Build a web UI with Gradio or Streamlit
- Combine with recommendation system (user purchase history + visual search)